# 이야기담 — COCO 사전학습 가중치에서 재학습 (손목 성능 개선)
**순서대로 실행.** 런타임: GPU (L4 약 9시간 / A100 약 5시간). 세션이 끊기면 §1~§6을 다시 실행하고 §9 셀들을 실행 → 이어서 학습.

**왜:** 배포 모델(v1)은 학습 첫 실행에 `--resume`을 붙여 `load_from`이 지워지는 바람에 **사전학습 없이 처음부터** 학습됐다. 포즈 모델 평가와 동작 평가 모두 **손목이 가장 약했다**(동작 평가 재검출 0.77~0.84).
**무엇을:** v1과 **완전히 같은 설정**(30에폭, 배치 256, lr 1e-3, 같은 split·augmentation)에서 **시작 가중치만** COCO 사전학습 RTMPose-m으로 바꾼다. 차이가 곧 사전학습의 효과.
**확인:** 같은 test 8,908장에서 coco/AP, PCK, **관절별(손목·팔꿈치)** 비교 → 나으면 deploy로 내보내고 동작 입히기 노트북에서 모델만 교체.

## 1. Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/yiyagidam'
os.makedirs(f'{DRIVE}/work_dirs', exist_ok=True)
CFG_DIR = '/content/mmpose/configs/body_2d_keypoint/rtmpose/coco/'
PY = 'MPLBACKEND=Agg PYTHONUNBUFFERED=1 /usr/local/miniconda/bin/conda run --no-capture-output -n mmpose_env python -u'

NAME   = 'drawing15_pre'                       # 이번 실험 이름 (config / work_dir / log 모두 이 이름)
PRE_CFG = CFG_DIR + NAME + '.py'
W      = f'{DRIVE}/work_dirs/{NAME}'
LOG    = f'{DRIVE}/train_{NAME}.log'

!nvidia-smi --query-gpu=name,memory.total --format=csv
print('CPU:', os.cpu_count())
!ls {DRIVE}/deploy

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
NVIDIA L4, 23034 MiB
CPU: 12
drawing15_full.py  drawing15_v2b.py		 rtmpose-m_drawing15_v2b.pth
drawing15.py	   rtmpose-m_drawing15_ep30.pth


## 2. conda 환경 + mmpose 설치 (~10분)
마지막 줄에 `CUDA True | mmpose import OK`가 찍혀야 함.

In [ ]:
%cd /content
!wget -q -nc https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!test -d /usr/local/miniconda || bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda > /dev/null
!/usr/local/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/usr/local/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!test -d /usr/local/miniconda/envs/mmpose_env || /usr/local/miniconda/bin/conda create -n mmpose_env python=3.10 -y -q

E = "/usr/local/miniconda/bin/conda run -n mmpose_env"
!{E} pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!{E} pip install -q "numpy<2" "opencv-python<5"
!{E} pip install -q -U openmim
!{E} mim install -q mmengine "mmcv==2.1.0" "mmdet>=3.1.0"
!test -d mmpose || git clone -q https://github.com/open-mmlab/mmpose.git
!{E} pip install -q "numpy<2" chumpy --no-build-isolation
!{E} bash -c "cd /content/mmpose && pip install -q -e ."
!{E} pip install -q "albumentations==1.3.1" future tensorboard
!{E} pip install -q "numpy<2" "opencv-python<5" "opencv-python-headless<5"
!cd / && MPLBACKEND=Agg /usr/local/miniconda/bin/conda run -n mmpose_env python -c "import torch; from mmpose.apis import init_model; print('CUDA', torch.cuda.is_available(), '| mmpose import OK')"

# 설치 캐시 정리 (학습 결과에 영향 없음, 수 GB 확보)
!/usr/local/miniconda/bin/conda clean -a -y -q
!{E} pip cache purge
!rm -f /content/Miniconda3-latest-Linux-x86_64.sh
!df -h /content

/content
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
mim resources not found: /usr/local/miniconda/envs/mmpose_env/lib/python3.10/site-packages/mmpose/.mim, you can try to install the latest mmpose.
CUDA True | mmpose import OK
Files removed: 8 (1.1 MB)
Directories removed: 21
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   18G   96G  16% /


## 3. 라벨 + COCO 사전학습 가중치 다운로드

In [ ]:
%cd /content
!wget -q -nc https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings_annotations.json
!wget -q -nc https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-coco_pt-aic-coco_420e-256x192-d8dd5ca4_20230127.pth -O rtmpose-m_coco-256x192.pth
!df -h /content

/content
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   18G   96G  16% /


## 4. 17→15 관절 변환 + split (**v1 split_ids.json 그대로** → test 8,908장 동일)

In [ ]:
import json, random
import numpy as np

COCO_IDX = {n: i for i, n in enumerate([
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"])}
J15 = ["nose", "neck", "right_shoulder", "right_elbow", "right_wrist",
       "left_shoulder", "left_elbow", "left_wrist", "pelvis",
       "right_hip", "right_knee", "right_ankle", "left_hip", "left_knee", "left_ankle"]
SKEL_1BASED = [[a + 1, b + 1] for a, b in [(0,1),(1,2),(2,3),(3,4),(1,5),(5,6),(6,7),
                                           (1,8),(8,9),(9,10),(10,11),(8,12),(12,13),(13,14)]]

def to15(k17):
    k = np.array(k17, dtype=np.float32).reshape(17, 3)
    g = lambda n: k[COCO_IDX[n]]
    mid = lambda a, b: np.array([(g(a)[0]+g(b)[0])/2, (g(a)[1]+g(b)[1])/2, min(g(a)[2], g(b)[2])])
    return np.stack([g("nose"), mid("left_shoulder", "right_shoulder"),
                     g("right_shoulder"), g("right_elbow"), g("right_wrist"),
                     g("left_shoulder"), g("left_elbow"), g("left_wrist"),
                     mid("left_hip", "right_hip"),
                     g("right_hip"), g("right_knee"), g("right_ankle"),
                     g("left_hip"), g("left_knee"), g("left_ankle")])

raw = json.load(open('/content/amateur_drawings_annotations.json'))
anns = []
for a in raw['annotations']:
    if a.get('num_keypoints', 0) == 0:
        continue
    k = to15(a['keypoints'])
    if (k[:, 2] > 0).sum() < 8:
        continue
    anns.append(dict(id=a['id'], image_id=a['image_id'], bbox=a['bbox'], kpts=k.tolist()))
img_by_id = {im['id']: im for im in raw['images']}
print('annotations:', len(anns))

# v1 때 저장해 둔 split을 그대로 불러온다 (test 8,908장 동일 보장)
SPLIT = {k: set(v) for k, v in json.load(open(f'{DRIVE}/split_ids.json')).items()}
assert not (SPLIT['train'] & SPLIT['val']) and not (SPLIT['train'] & SPLIT['test']) and not (SPLIT['val'] & SPLIT['test'])
valid_ids = {a['image_id'] for a in anns}
for _k in ['train', 'val', 'test']:
    print(_k, '어노테이션 없는 id:', len(SPLIT[_k] - valid_ids))   # 0이어야 정상
print({k: len(v) for k, v in SPLIT.items()})

annotations: 178166
train 어노테이션 없는 id: 0
val 어노테이션 없는 id: 0
test 어노테이션 없는 id: 0
{'test': 8908, 'val': 8908, 'train': 160350, 'smoke_train': 3000, 'smoke_val': 500}


## 5. 이미지 스트리밍 다운로드 → 바로 리사이즈(긴 변 ≤1024, JPEG) + COCO json 생성 (20~40분)
원본 이미지를 디스크에 저장하지 않는다. tar를 받으면서 한 장씩 꺼내 메모리에서 리사이즈한 뒤 `data_small`에만 저장 → 디스크 최대 사용량 = `data_small` 크기.
리사이즈 방식(BILINEAR, quality 95, 좌표 스케일)은 v1과 완전히 동일.
중간에 끊기면 이 셀을 다시 실행: `_info.jsonl`에 기록된 이미지는 건너뛴다(다운로드는 처음부터 다시 흘러가지만 디스크는 안 늘어남).

In [ ]:
import os, io, json, tarfile, subprocess
from PIL import Image
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED

URL = 'https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings.tar'
DST, MAX = '/content/data_small/', 1024
INFO_PATH = DST + '_info.jsonl'
used = set().union(*[SPLIT[s] for s in ['train', 'val', 'test']])
id_by_name = {img_by_id[i]['file_name']: i for i in used}   # 같은 파일을 가리키는 id가 여럿이면 하나만 남음 → 아래에서 복사로 채움

def proc_bytes(iid, fn0, data):           # v1의 proc()와 같은 처리, 입력만 파일 대신 bytes
    fn = os.path.splitext(fn0)[0] + '.jpg'
    img = Image.open(io.BytesIO(data))
    w, h = img.size
    s = min(1.0, MAX / max(w, h))
    os.makedirs(os.path.dirname(DST + fn), exist_ok=True)
    img = img.convert('RGB')
    if s < 1.0:
        img = img.resize((round(w * s), round(h * s)), Image.BILINEAR)
    img.save(DST + fn, quality=95)
    return iid, (fn, s, round(w * s), round(h * s))

# 이전에 처리한 것 불러오기 (재실행 대비)
os.makedirs(DST, exist_ok=True)
INFO = {}
if os.path.exists(INFO_PATH):
    for l in open(INFO_PATH):
        try:
            iid, v = json.loads(l); INFO[iid] = tuple(v)
        except json.JSONDecodeError:
            pass                           # 끊길 때 반쯤 쓴 마지막 줄은 무시
todo = used - set(INFO)
print(f'이미 처리 {len(INFO)} / 남음 {len(todo)}')

if todo:
    wget = subprocess.Popen(['wget', '-q', '-O', '-', URL], stdout=subprocess.PIPE)
    seen = matched = 0
    with ProcessPoolExecutor(os.cpu_count()) as ex, \
         tarfile.open(fileobj=wget.stdout, mode='r|*') as tar, \
         open(INFO_PATH, 'a') as log:
        pending = set()
        def collect(fs):
            for f in fs:
                iid, v = f.result()
                INFO[iid] = v
                log.write(json.dumps([iid, v]) + '\n')
        for m in tar:
            if not m.isfile():
                continue
            seen += 1
            name = m.name[2:] if m.name.startswith('./') else m.name
            iid = id_by_name.get(name)
            if seen == 2000 and matched == 0:
                raise RuntimeError(f'tar 경로와 file_name이 안 맞음. 예: tar={name!r} / json={next(iter(id_by_name))!r}')
            if iid is None or iid not in todo:
                continue
            matched += 1
            pending.add(ex.submit(proc_bytes, iid, name, tar.extractfile(m).read()))
            if len(pending) >= 256:        # 메모리에 쌓이는 이미지 수 제한
                done, pending = wait(pending, return_when=FIRST_COMPLETED)
                collect(done)
            if matched % 10000 == 0:
                log.flush()
                print(f'{len(INFO)} / {len(used)}', flush=True)
        collect(wait(pending)[0])
    wget.wait()

# 라벨 json에 같은 파일을 가리키는 id가 여럿 있음 (443건) → 처리된 결과를 그대로 복사 (같은 파일 → 같은 결과)
by_name = {}
for i in INFO:
    by_name.setdefault(img_by_id[i]['file_name'], INFO[i])
with open(INFO_PATH, 'a') as log:
    for i in used - set(INFO):
        v = by_name.get(img_by_id[i]['file_name'])
        if v:
            INFO[i] = tuple(v); log.write(json.dumps([i, v]) + '\n')

missing = used - set(INFO)
assert not missing, f'{len(missing)}장 누락 (다운로드가 중간에 끊겼으면 이 셀 재실행)'
print('resized:', len(INFO))
!du -sh {DST}; df -h /content

def write_coco(split, dst):
    ids = SPLIT[split]
    out_anns = []
    for a in anns:
        if a['image_id'] not in ids:
            continue
        s = INFO[a['image_id']][1]
        flat, nv = [], 0
        for x, y, v in a['kpts']:
            flat += [x * s, y * s, int(v)]
            nv += int(v > 0)
        bx = [v * s for v in a['bbox']]
        out_anns.append(dict(id=a['id'], image_id=a['image_id'], category_id=1, bbox=bx,
                             area=bx[2] * bx[3], iscrowd=0, keypoints=flat, num_keypoints=nv))
    imgs = [dict(id=i, file_name=INFO[i][0], width=INFO[i][2], height=INFO[i][3]) for i in ids]
    json.dump(dict(images=imgs, annotations=out_anns,
                   categories=[dict(id=1, name='human', supercategory='drawing',
                                    keypoints=J15, skeleton=SKEL_1BASED)]), open(dst, 'w'))
    print(f'{dst}: images {len(imgs)} / anns {len(out_anns)}')

for sp in SPLIT:
    write_coco(sp, f'/content/{sp}_coco.json')

이미 처리 178166 / 남음 0
resized: 178166
9.7G	/content/data_small/
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   18G   96G  16% /
/content/test_coco.json: images 8908 / anns 8908
/content/val_coco.json: images 8908 / anns 8908
/content/train_coco.json: images 160350 / anns 160350
/content/smoke_train_coco.json: images 3000 / anns 3000
/content/smoke_val_coco.json: images 500 / anns 500


## 6. dataset_info + v1 config 복원 (v1 비교용)

In [ ]:
%%writefile /content/mmpose/configs/_base_/datasets/drawing15.py
dataset_info = dict(
    dataset_name='drawing15',
    paper_info=dict(),
    keypoint_info={
        0: dict(name='nose', id=0, color=[255, 128, 0], type='upper', swap='', sigma=0.026),
        1: dict(name='neck', id=1, color=[255, 128, 0], type='upper', swap='', sigma=0.079),
        2: dict(name='right_shoulder', id=2, color=[255, 128, 0], type='upper', swap='left_shoulder', sigma=0.079),
        3: dict(name='right_elbow', id=3, color=[255, 128, 0], type='upper', swap='left_elbow', sigma=0.072),
        4: dict(name='right_wrist', id=4, color=[255, 128, 0], type='upper', swap='left_wrist', sigma=0.062),
        5: dict(name='left_shoulder', id=5, color=[0, 255, 0], type='upper', swap='right_shoulder', sigma=0.079),
        6: dict(name='left_elbow', id=6, color=[0, 255, 0], type='upper', swap='right_elbow', sigma=0.072),
        7: dict(name='left_wrist', id=7, color=[0, 255, 0], type='upper', swap='right_wrist', sigma=0.062),
        8: dict(name='pelvis', id=8, color=[255, 128, 0], type='lower', swap='', sigma=0.107),
        9: dict(name='right_hip', id=9, color=[255, 128, 0], type='lower', swap='left_hip', sigma=0.107),
        10: dict(name='right_knee', id=10, color=[255, 128, 0], type='lower', swap='left_knee', sigma=0.087),
        11: dict(name='right_ankle', id=11, color=[255, 128, 0], type='lower', swap='left_ankle', sigma=0.089),
        12: dict(name='left_hip', id=12, color=[0, 255, 0], type='lower', swap='right_hip', sigma=0.107),
        13: dict(name='left_knee', id=13, color=[0, 255, 0], type='lower', swap='right_knee', sigma=0.087),
        14: dict(name='left_ankle', id=14, color=[0, 255, 0], type='lower', swap='right_ankle', sigma=0.089),
    },
    skeleton_info={
        0: dict(link=('nose', 'neck'), id=0, color=[0, 255, 255]),
        1: dict(link=('neck', 'right_shoulder'), id=1, color=[255, 128, 0]),
        2: dict(link=('right_shoulder', 'right_elbow'), id=2, color=[255, 128, 0]),
        3: dict(link=('right_elbow', 'right_wrist'), id=3, color=[255, 128, 0]),
        4: dict(link=('neck', 'left_shoulder'), id=4, color=[0, 255, 0]),
        5: dict(link=('left_shoulder', 'left_elbow'), id=5, color=[0, 255, 0]),
        6: dict(link=('left_elbow', 'left_wrist'), id=6, color=[0, 255, 0]),
        7: dict(link=('neck', 'pelvis'), id=7, color=[0, 255, 255]),
        8: dict(link=('pelvis', 'right_hip'), id=8, color=[255, 128, 0]),
        9: dict(link=('right_hip', 'right_knee'), id=9, color=[255, 128, 0]),
        10: dict(link=('right_knee', 'right_ankle'), id=10, color=[255, 128, 0]),
        11: dict(link=('pelvis', 'left_hip'), id=11, color=[0, 255, 0]),
        12: dict(link=('left_hip', 'left_knee'), id=12, color=[0, 255, 0]),
        13: dict(link=('left_knee', 'left_ankle'), id=13, color=[0, 255, 0]),
    },
    joint_weights=[1.] * 15,
    sigmas=[0.026, 0.079, 0.079, 0.072, 0.062, 0.079, 0.072, 0.062,
            0.107, 0.107, 0.087, 0.089, 0.107, 0.087, 0.089])

Overwriting /content/mmpose/configs/_base_/datasets/drawing15.py


In [ ]:
import shutil, os
D = f'{DRIVE}/deploy'
assert os.path.exists(f'{D}/rtmpose-m_drawing15_ep30.pth'), 'v1 체크포인트 없음'
shutil.copy(f'{D}/drawing15_full.py', CFG_DIR + 'drawing15_full.py')   # v1 config
print(os.listdir(CFG_DIR)[:5], '...')

['rtmpose-s_8xb256-420e_coco-256x192.py', 'rtmpose-t_8xb256-420e_coco-256x192.py', 'rtmpose_coco.yml', 'rtmpose-m_8xb256-420e_coco-256x192.py', 'rtmpose-l_8xb256-420e_aic-coco-256x192.py'] ...


## 7. 시작 가중치 준비 (COCO 17관절 → 출력층만 제거)

In [ ]:
%%writefile /content/strip_ckpt.py
"""COCO 사전학습 RTMPose-m(17관절)에서 마지막 출력층만 떼어 낸 시작 가중치 (우리 그림은 15관절이라 출력층만 새로 학습)"""
import torch
sd = torch.load('/content/rtmpose-m_coco-256x192.pth', map_location='cpu')['state_dict']
drop = [k for k in sd if k.startswith('head.final_layer')]
for k in drop:
    sd.pop(k)
torch.save({'state_dict': sd}, '/content/rtmpose-m_init15.pth')
print(f'ok | 뗀 층 {drop} | 남은 가중치 {len(sd)}개')

Overwriting /content/strip_ckpt.py


In [ ]:
!{PY} /content/strip_ckpt.py

ok | 뗀 층 ['head.final_layer.weight', 'head.final_layer.bias'] | 남은 가중치 360개


## 8. 학습 config (v1과 같고 시작 가중치만 다름) + 검증

In [ ]:
%%writefile /content/make_pre_cfg.py
"""v1과 완전히 같은 설정(30에폭, 배치 256, lr 1e-3, augmentation 동일)에서 시작 가중치만 COCO 사전학습으로"""
from mmengine.config import Config
D = '/content/drive/MyDrive/yiyagidam'
C = '/content/mmpose/configs/body_2d_keypoint/rtmpose/coco/'
NAME = 'drawing15_pre'
cfg = Config.fromfile(f'{D}/deploy/drawing15_full.py')
cfg.load_from, cfg.resume = '/content/rtmpose-m_init15.pth', False      # --resume은 last_checkpoint가 있을 때만 스크립트가 붙임
cfg.work_dir = f'{D}/work_dirs/{NAME}'
cfg.train_dataloader.dataset.ann_file = 'train_coco.json'
cfg.val_dataloader.dataset.ann_file = 'val_coco.json'
cfg.val_evaluator.ann_file = '/content/val_coco.json'
cfg.test_dataloader.dataset.ann_file = 'test_coco.json'
cfg.test_evaluator.ann_file = '/content/test_coco.json'
cfg.dump(C + NAME + '.py')
c = Config.fromfile(C + NAME + '.py')                                     # 저장된 파일로 다시 검증
assert c.load_from.endswith('init15.pth') and c.resume is False and c.train_cfg.max_epochs == 30
assert c.train_dataloader.dataset.ann_file == 'train_coco.json'
print(NAME, '| epochs', c.train_cfg.max_epochs, '| batch', c.train_dataloader.batch_size,
      '| lr', c.optim_wrapper.optimizer.lr, '| load_from', c.load_from, '| work_dir', c.work_dir)

Overwriting /content/make_pre_cfg.py


In [ ]:
!{PY} /content/make_pre_cfg.py

drawing15_pre | epochs 30 | batch 256 | lr 0.001 | load_from /content/rtmpose-m_init15.pth | work_dir /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre


## 9. 학습 (백그라운드)
- 첫 실행: `--resume` 없이 시작 → **COCO 사전학습 가중치를 불러옴** (v1에서 문제였던 부분)
- 세션이 끊긴 뒤: §1~§6, 그리고 이 섹션의 스크립트 셀을 다시 실행하고 실행 셀을 실행 → `last_checkpoint`가 있으면 자동으로 이어서 학습
- 실행 셀은 **한 번만** (두 번 누르면 학습이 두 개 뜸)

In [ ]:
%%writefile /content/train_v2.sh
#!/bin/bash
# 사용법: bash train_v2.sh <config> <work_dir>
# last_checkpoint가 있을 때만 --resume (첫 실행은 load_from=v1 가중치로 시작)
CFG="$1"; W="$2"
for i in 1 2 3; do
  R=""
  [ -f "$W/last_checkpoint" ] && R="--resume"
  echo "[train_v2.sh] try $i: $R"
  env MPLBACKEND=Agg PYTHONUNBUFFERED=1 /usr/local/miniconda/bin/conda run --no-capture-output -n mmpose_env \
      python -u /content/mmpose/tools/train.py "$CFG" $R && break
  sleep 60
done

Overwriting /content/train_v2.sh


In [ ]:
if os.path.exists(f'{W}/last_checkpoint'):
    print('이어서 학습:', open(f'{W}/last_checkpoint').read().strip())
else:
    print('새로 시작: COCO 사전학습 가중치(rtmpose-m_init15.pth)에서 출발')
!nohup bash /content/train_v2.sh {PRE_CFG} {W} >> {LOG} 2>&1 &

이어서 학습: /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/epoch_30.pth


In [ ]:
!df -h /content/drive/MyDrive

Filesystem      Size  Used Avail Use% Mounted on
drive            15G   13G  2.9G  82% /content/drive


In [ ]:
# 모니터링 (반복 실행)
# 시작 직후 꼭 확인: "Load checkpoint from /content/rtmpose-m_init15.pth" 가 찍혔는지 (v1은 이 줄이 없었음)
# 출력층(head.final_layer)이 없다는 경고는 정상 (15관절용으로 새로 학습하는 층)
!pgrep -af "python -u /content/mmpose/tools/train.py" | grep -v pgrep | head -2
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!grep -E "Load checkpoint from|Auto resumed|Resumed checkpoint" {LOG} | tail -2
!grep "Epoch(train)" {LOG} | tail -1 | cut -c1-230
!grep "Epoch(val)" {LOG} | tail -3 | cut -c1-120

0 %, 0 MiB
11/09 05:03:57 - mmengine - INFO - Auto resumed from the latest checkpoint /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/epoch_30.pth.
11/09 05:03:58 - mmengine - INFO - Load checkpoint from /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/epoch_30.pth
11/09 05:00:01 - mmengine - INFO - Epoch(train) [30][600/627]  base_lr: 5.002078e-05 lr: 5.002078e-05  eta: 0:00:41  time: 1.515858  data_time: 0.535147  memory: 16874  loss: 1.306334  loss_kpt: 1.306334  acc_pose: 0.894524
11/09 04:28:07 - mmengine - INFO - Epoch(val) [28][35/35]    coco/AP: 0.920393  coco/AP .5: 0.988871  coco/AP .75: 0.944
11/09 04:44:40 - mmengine - INFO - Epoch(val) [29][35/35]    coco/AP: 0.920515  coco/AP .5: 0.988865  coco/AP .75: 0.944
11/09 05:01:12 - mmengine - INFO - Epoch(val) [30][35/35]    coco/AP: 0.921528  coco/AP .5: 0.988878  coco/AP .75: 0.944


In [ ]:
# # (필요할 때만) 학습 강제 중지
# !pkill -f "train_v2.sh"
# !pkill -f "tools/train.py"
# !pgrep -af "tools/train.py" | grep -v pgrep

In [ ]:
# 에폭별 lr / train loss / val AP 요약 (lr이 cosine으로 줄어드는지 확인)
import re
EPOCHS = 30
lines = open(LOG, errors='ignore').read().splitlines()
lr, loss, val = {}, {}, {}
for l in lines:
    m = re.search(r'Epoch\(train\)\s+\[(\d+)\]\[\s*\d+/\d+\]\s+base_lr: \S+\s+lr: ([\d.e+-]+).*?\sloss: ([\d.]+)', l)
    if m: lr[int(m[1])], loss[int(m[1])] = float(m[2]), float(m[3])   # 에폭의 마지막 줄 값
    m = re.search(r'Epoch\(val\)\s+\[(\d+)\]\[\d+/\d+\]\s+coco/AP: ([\d.]+)', l)
    if m: val[int(m[1])] = float(m[2])                                  # 재시작 중복은 마지막 값
nan = float('nan')
print('epoch |     lr     | train loss | val AP')
for e in range(1, EPOCHS + 1):
    print(f'{e:5d} | {lr.get(e, nan):.3e} | {loss.get(e, nan):10.4f} | {val.get(e, nan):.4f}')
!ls -lt {W}/best_coco_AP_*.pth

epoch |     lr     | train loss | val AP
    1 | 1.000e-03 |     1.9372 | 0.8927
    2 | 1.000e-03 |     1.6977 | 0.9077
    3 | 1.000e-03 |     1.6279 | 0.9112
    4 | 1.000e-03 |     1.5954 | 0.9142
    5 | 1.000e-03 |     1.5827 | 0.9118
    6 | 1.000e-03 |     1.5689 | 0.9160
    7 | 1.000e-03 |     1.5481 | 0.9158
    8 | 1.000e-03 |     1.5618 | 0.9158
    9 | 1.000e-03 |     1.5489 | 0.9166
   10 | 1.000e-03 |     1.5548 | 0.9156
   11 | 1.000e-03 |     1.5362 | 0.9167
   12 | 1.000e-03 |     1.5430 | 0.9168
   13 | 1.000e-03 |     1.5291 | 0.9166
   14 | 1.000e-03 |     1.5345 | 0.9167
   15 | 1.000e-03 |     1.5328 | 0.9169
   16 | 9.905e-04 |     1.5107 | 0.9172
   17 | 9.607e-04 |     1.5212 | 0.9160
   18 | 9.119e-04 |     1.5172 | 0.9170
   19 | 8.461e-04 |     1.5051 | 0.9169
   20 | 7.663e-04 |     1.4901 | 0.9169
   21 | 6.760e-04 |     1.4850 | 0.9169
   22 | 5.791e-04 |     1.4686 | 0.9179
   23 | 4.798e-04 |     1.4629 | 0.9180
   24 | 3.824e-04 |     1.4472 | 0.9180

## 10. v1과 비교 (같은 test 8,908장)
학습이 끝나 `work_dirs/drawing15_pre/best_coco_AP_*.pth`가 생긴 뒤 실행. 비교 스크립트 출력의 `v2` = 이번 모델(pre).

In [ ]:
%%writefile /content/compare_models.py
import os; os.environ['MPLBACKEND'] = 'Agg'
import sys, json, glob, numpy as np
from mmpose.apis import init_model, inference_topdown

D = '/content/drive/MyDrive/yiyagidam'; C = '/content/mmpose/configs/body_2d_keypoint/rtmpose/coco/'
latest = lambda pat: max(glob.glob(pat), key=os.path.getmtime)
NAME = sys.argv[1]
MODELS = {'v1_ep30': (C + 'drawing15_full.py', f'{D}/deploy/rtmpose-m_drawing15_ep30.pth'),
          'v2':      (C + f'{NAME}.py',         latest(f'{D}/work_dirs/{NAME}/best_coco_AP_*.pth'))}
VARIANTS = {'base': {}, 'no_flip': {'model.test_cfg.flip_test': False},
            'dark': {'model.head.decoder.use_dark': True}}
NAMES = ["nose","neck","r_sho","r_elb","r_wri","l_sho","l_elb","l_wri","pelvis","r_hip","r_knee","r_ank","l_hip","l_knee","l_ank"]
ORIG13 = [i for i in range(15) if i not in (1, 8)]

coco = json.load(open('/content/test_coco.json'))
files = {im['id']: im['file_name'] for im in coco['images']}; anns = coco['annotations']

def run(cfg, ckpt, opts):
    m = init_model(cfg, ckpt, device='cuda:0', cfg_options=opts or None)
    d = np.full((len(anns), 15), np.nan)
    for n, a in enumerate(anns):
        x, y, w, h = a['bbox']
        r = inference_topdown(m, '/content/data_small/' + files[a['image_id']],
                              bboxes=np.array([[x, y, x + w, y + h]]), bbox_format='xyxy')[0]
        p = r.pred_instances.keypoints[0]
        gt = np.array(a['keypoints'], dtype=np.float32).reshape(15, 3)
        dist = np.linalg.norm(p - gt[:, :2], axis=1) / max(w, h)
        d[n] = np.where(gt[:, 2] > 0, dist, np.nan)
    return d

res = {}
for mn, (cfg, ck) in MODELS.items():
    print(mn, ck)
    for vn, ov in VARIANTS.items():
        try:
            res[(mn, vn)] = run(cfg, ck, ov); print('  done', vn)
        except Exception as e:
            print('  skip', vn, '->', repr(e)[:200])

rows = ['model,variant,PCK@0.05_mean13,PCK@0.1_mean13,' + ','.join(f'{n}@0.05' for n in NAMES)]
for (mn, vn), d in res.items():
    vis = ~np.isnan(d)
    j05 = ((d < 0.05) & vis).sum(0) / vis.sum(0); j10 = ((d < 0.1) & vis).sum(0) / vis.sum(0)
    rows.append(f'{mn},{vn},{j05[ORIG13].mean():.4f},{j10[ORIG13].mean():.4f},' + ','.join(f'{v:.3f}' for v in j05))
out = '\n'.join(rows); print(out); open(f'{D}/compare_v1_{NAME}.csv', 'w').write(out)

# 부트스트랩: v2 - v1 (base) PCK@0.05 mean13 차이의 95% 신뢰구간
def per_inst(d):
    d = d[:, ORIG13]; ok = ~np.isnan(d)
    return np.where(ok.sum(1) > 0, ((d < 0.05) & ok).sum(1) / np.maximum(ok.sum(1), 1), np.nan)
sa, sb = per_inst(res[('v1_ep30', 'base')]), per_inst(res[('v2', 'base')])
rng = np.random.default_rng(0); diffs = []
for _ in range(2000):
    i = rng.integers(0, len(sa), len(sa)); diffs.append(np.nanmean(sb[i]) - np.nanmean(sa[i]))
print(f'v2-v1 PCK@0.05 diff: {np.mean(diffs):+.4f}  95% CI [{np.percentile(diffs,2.5):+.4f}, {np.percentile(diffs,97.5):+.4f}]  (CI가 0을 포함하면 노이즈)')

Writing /content/compare_models.py


In [ ]:
!{PY} /content/compare_models.py {NAME}

v1_ep30 /content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth
  done base
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth
  done no_flip
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth
  done dark
v2 /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/best_coco_AP_epoch_30.pth
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/best_coco_AP_epoch_30.pth
  done base
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/best_coco_AP_epoch_30.pth
  done no_flip
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/best_coco_AP_epoch_30.pth
  done dark
model,variant,PCK@0.05_mean13,PCK

### 관절별 비교 (손목·팔꿈치 먼저)

In [ ]:
# 관절별 PCK@0.05 (test 8,908장, 기본 추론): 손목·팔꿈치 중심으로 v1과 비교
import pandas as pd
df = pd.read_csv(f'{DRIVE}/compare_v1_{NAME}.csv')
base = df[df.variant == 'base'].set_index('model')
cols = [c for c in base.columns if c.endswith('@0.05') and not c.startswith('PCK')]
t = base[cols].T
t.columns = ['v1 (처음부터)', 'pre (COCO 사전학습)']
t['차이'] = (t.iloc[:, 1] - t.iloc[:, 0]).round(3)
t.index = [c.replace('@0.05', '') for c in t.index]
order = ['r_wri', 'l_wri', 'r_elb', 'l_elb'] + [c for c in t.index if c not in ('r_wri', 'l_wri', 'r_elb', 'l_elb')]
display(t.loc[order])
print(base[['PCK@0.05_mean13', 'PCK@0.1_mean13']])

,v1 (처음부터),pre (COCO 사전학습),차이
r_wri,0.878,0.881,0.003
l_wri,0.865,0.875,0.010
r_elb,0.905,0.909,0.004
l_elb,0.898,0.906,0.008
nose,0.934,0.936,0.002
neck,0.934,0.937,0.003
r_sho,0.927,0.929,0.002
l_sho,0.925,0.930,0.005
pelvis,0.924,0.929,0.005
r_hip,0.920,0.923,0.003


         PCK@0.05_mean13  PCK@0.1_mean13
model                                   
v1_ep30           0.9099          0.9496
v2                0.9149          0.9514


### coco AP (공식 지표)

In [ ]:
CK_V1 = f'{DRIVE}/deploy/rtmpose-m_drawing15_ep30.pth'
print('v1 (처음부터 학습)')
!{PY} /content/mmpose/tools/test.py {CFG_DIR}drawing15_full.py {CK_V1} 2>&1 | grep -E "Epoch\(test\)|Error"
CK_PRE = !ls -t {W}/best_coco_AP_*.pth | head -1
print('pre (COCO 사전학습):', CK_PRE[0])
!{PY} /content/mmpose/tools/test.py {PRE_CFG} {CK_PRE[0]} 2>&1 | grep -E "Epoch\(test\)|Error"

v1 (처음부터 학습)
11/09 05:37:08 - mmengine - INFO - Epoch(test) [35/35]    coco/AP: 0.919476  coco/AP .5: 0.978759  coco/AP .75: 0.945408  coco/AP (M): 0.930708  coco/AP (L): 0.919457  coco/AR: 0.934205  coco/AR .5: 0.988886  coco/AR .75: 0.953637  coco/AR (M): 0.940000  coco/AR (L): 0.934156  data_time: 0.200626  time: 0.789756
pre (COCO 사전학습): /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/best_coco_AP_epoch_30.pth
11/09 05:37:53 - mmengine - INFO - Epoch(test) [35/35]    coco/AP: 0.922246  coco/AP .5: 0.978890  coco/AP .75: 0.946076  coco/AP (M): 0.936435  coco/AP (L): 0.922237  coco/AR: 0.936798  coco/AR .5: 0.988774  coco/AR .75: 0.957342  coco/AR (M): 0.945333  coco/AR (L): 0.936726  data_time: 0.201077  time: 0.782635


## 11. deploy로 내보내기
**§10에서 v1보다 나을 때만.** `deploy/rtmpose-m_drawing15_pre.pth`, `deploy/drawing15_pre.py`로 저장 (v1 파일은 그대로).

In [ ]:
%%writefile /content/export_v2.py
import sys, torch, shutil, os, glob
NAME = sys.argv[1]
D = '/content/drive/MyDrive/yiyagidam'
ck = sorted(glob.glob(f'{D}/work_dirs/{NAME}/best_coco_AP_*.pth'), key=os.path.getmtime)[-1]
c = torch.load(ck, map_location='cpu')
torch.save({'state_dict': c['state_dict'], 'meta': c.get('meta', {})}, f'{D}/deploy/rtmpose-m_{NAME}.pth')
shutil.copy(f'/content/mmpose/configs/body_2d_keypoint/rtmpose/coco/{NAME}.py', f'{D}/deploy/{NAME}.py')
print('saved from', ck)
print(sorted(os.listdir(f'{D}/deploy')))

Writing /content/export_v2.py


In [ ]:
!{PY} /content/export_v2.py {NAME}
!ls -lh {DRIVE}/deploy/

saved from /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_pre/best_coco_AP_epoch_30.pth
['drawing15.py', 'drawing15_full.py', 'drawing15_pre.py', 'drawing15_v2b.py', 'rtmpose-m_drawing15_ep30.pth', 'rtmpose-m_drawing15_pre.pth', 'rtmpose-m_drawing15_v2b.pth']
total 156M
-rw------- 1 root root  13K Oct 25 06:37 drawing15_full.py
-rw------- 1 root root  13K Nov  9 05:37 drawing15_pre.py
-rw------- 1 root root 2.9K Oct 25 06:37 drawing15.py
-rw------- 1 root root  13K Oct 26 11:07 drawing15_v2b.py
-rw------- 1 root root  52M Oct 25 06:37 rtmpose-m_drawing15_ep30.pth
-rw------- 1 root root  52M Nov  9 05:37 rtmpose-m_drawing15_pre.pth
-rw------- 1 root root  52M Oct 26 11:07 rtmpose-m_drawing15_v2b.pth


## 12. 동작 입히기 노트북에서 새 모델 쓰기
아래 셀은 **동작 입히기 노트북에서** 실행 (이 노트북 아님). 그 뒤 동작 입히기·평가를 다시 돌려 손목 점수 비교.

In [ ]:
# ▼ 동작 입히기 노트북(동작_입히기_BVH)에서 새 모델을 쓰려면: 그 노트북의 2번 셀(애니메이션 스크립트) 실행 후 아래를 실행
# (배경 분리·동작 입히기·평가가 모두 새 모델로 바뀜)
from pathlib import Path
import shutil
D = '/content/drive/MyDrive/yiyagidam/deploy'
shutil.copy(f'{D}/drawing15_pre.py', '/content/mmpose/configs/body_2d_keypoint/rtmpose/coco/drawing15_pre.py')
p = Path('/content/animate_drawing.py'); s = p.read_text(encoding='utf-8')
s = s.replace("rtmpose/coco/drawing15_full.py'", "rtmpose/coco/drawing15_pre.py'").replace("rtmpose-m_drawing15_ep30.pth", "rtmpose-m_drawing15_pre.pth")
p.write_text(s, encoding='utf-8')
print([l for l in s.splitlines() if l.startswith(('CFG =', 'CKPT ='))])